# Final Project – BeautifulSoup Documentation Analytics System

**Project goal:** collect, parse, extract, analyze, and visualize information from the official BeautifulSoup documentation page.

**Target URL:** https://www.crummy.com/software/BeautifulSoup/bs4/doc/

This notebook is organized to satisfy the final project requirements:

1. Web Page Collector  
2. HTML Parser  
3. Section Extractor  
4. Link Extractor  
5. Code Example Extractor  
6. Documentation Analytics  
7. Data Visualization  
8. Final Report Content

## 1. Import Required Libraries

In [ ]:
import os
import re
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urljoin

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 2. Create Project Folders

In [ ]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("output/charts", exist_ok=True)

print("Folders created successfully.")


## 3. Feature 1 – Web Page Collector

This step downloads the BeautifulSoup documentation page using the `requests` library and saves the raw HTML file into `data/raw/beautifulsoup_doc.html`.

In [ ]:
url = "https://www.crummy.com/software/BeautifulSoup/bs4/doc/"
raw_html_path = "data/raw/beautifulsoup_doc.html"

response = requests.get(url, timeout=20)
print("Status code:", response.status_code)

if response.status_code == 200:
    with open(raw_html_path, "w", encoding="utf-8") as file:
        file.write(response.text)
    print("Raw HTML saved to:", raw_html_path)
else:
    raise Exception("Failed to download the documentation page.")


## 4. Feature 2 – HTML Parser

This step reads the saved HTML file and parses it using BeautifulSoup.

In [ ]:
with open(raw_html_path, "r", encoding="utf-8") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")

page_title = soup.title.get_text(strip=True) if soup.title else "No title found"
print("Page title:", page_title)


## 5. Helper Functions

These functions help identify sections, clean text, classify links, and find the section title for each extracted item.

In [ ]:
heading_tags = ["h1", "h2", "h3"]


def clean_text(text):
    """Clean extra spaces and line breaks from text."""
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def classify_link(href):
    """Classify link type based on href value."""
    if href is None or str(href).strip() == "":
        return "empty_or_invalid"

    href = str(href).strip()
    href_lower = href.lower()

    if href.startswith("#"):
        return "internal_anchor"

    if href_lower.endswith((".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp")):
        return "image_link"

    if "crummy.com/software/beautifulsoup/bs4/doc" in href_lower or "beautifulsoup" in href_lower:
        return "documentation_link"

    if href.startswith("http://") or href.startswith("https://"):
        return "external_link"

    return "empty_or_invalid"


def get_content_until_next_heading(heading):
    """Return all sibling elements after a heading until the next h1, h2, or h3."""
    content = []
    for sibling in heading.find_next_siblings():
        if sibling.name in heading_tags:
            break
        content.append(sibling)
    return content


## 6. Feature 3 – Section Extractor

Required output file: `data/processed/sections.csv`

Required columns:

- `section_id`
- `section_level`
- `section_title`
- `section_text`
- `word_count`
- `code_block_count`
- `link_count`

In [ ]:
headings = soup.find_all(heading_tags)
print("Number of headings found:", len(headings))

sections_data = []

for index, heading in enumerate(headings, start=1):
    section_title = clean_text(heading.get_text(" ", strip=True))
    section_level = heading.name
    section_elements = get_content_until_next_heading(heading)

    section_text_parts = []
    code_block_count = 0
    link_count = 0

    for element in section_elements:
        section_text_parts.append(clean_text(element.get_text(" ", strip=True)))
        code_block_count += len(element.find_all(["pre", "code"]))
        link_count += len(element.find_all("a"))

    section_text = clean_text(" ".join(section_text_parts))
    word_count = len(section_text.split())

    sections_data.append({
        "section_id": index,
        "section_level": section_level,
        "section_title": section_title,
        "section_text": section_text,
        "word_count": word_count,
        "code_block_count": code_block_count,
        "link_count": link_count
    })

sections_df = pd.DataFrame(sections_data)
sections_path = "data/processed/sections.csv"
sections_df.to_csv(sections_path, index=False, encoding="utf-8-sig")

print("Sections saved to:", sections_path)
sections_df.head()


## 7. Feature 4 – Link Extractor

Required output file: `data/processed/links.csv`

Required columns:

- `link_text`
- `href`
- `link_type`
- `section_title`

In [ ]:
links_data = []

for heading in headings:
    section_title = clean_text(heading.get_text(" ", strip=True))
    section_elements = get_content_until_next_heading(heading)

    for element in section_elements:
        for a_tag in element.find_all("a"):
            link_text = clean_text(a_tag.get_text(" ", strip=True))
            href = a_tag.get("href")
            absolute_href = urljoin(url, href) if href else ""

            links_data.append({
                "link_text": link_text,
                "href": href,
                "link_type": classify_link(href),
                "section_title": section_title,
                "absolute_href": absolute_href
            })

links_df = pd.DataFrame(links_data)
links_df = links_df[["link_text", "href", "link_type", "section_title", "absolute_href"]]

links_path = "data/processed/links.csv"
links_df.to_csv(links_path, index=False, encoding="utf-8-sig")

print("Links saved to:", links_path)
links_df.head()


## 8. Feature 5 – Code Example Extractor

Required output file: `data/processed/code_examples.csv`

Required columns:

- `example_id`
- `section_title`
- `code_text`
- `line_count`
- `contains_find_all`
- `contains_find`
- `contains_select`
- `contains_get_text`
- `contains_requests`

In [ ]:
code_examples_data = []
example_id = 1

for heading in headings:
    section_title = clean_text(heading.get_text(" ", strip=True))
    section_elements = get_content_until_next_heading(heading)

    for element in section_elements:
        code_blocks = element.find_all(["pre", "code"])

        for code in code_blocks:
            code_text = code.get_text("
", strip=True).strip()

            if code_text:
                code_examples_data.append({
                    "example_id": example_id,
                    "section_title": section_title,
                    "code_text": code_text,
                    "line_count": len(code_text.splitlines()),
                    "contains_find_all": "find_all" in code_text,
                    "contains_find": "find(" in code_text,
                    "contains_select": "select" in code_text,
                    "contains_get_text": "get_text" in code_text,
                    "contains_requests": "requests" in code_text
                })
                example_id += 1

code_examples_df = pd.DataFrame(code_examples_data)
code_examples_path = "data/processed/code_examples.csv"
code_examples_df.to_csv(code_examples_path, index=False, encoding="utf-8-sig")

print("Code examples saved to:", code_examples_path)
code_examples_df.head()


## 9. Load Processed Data

This step reloads the generated CSV files using Pandas.

In [ ]:
sections_df = pd.read_csv("data/processed/sections.csv")
links_df = pd.read_csv("data/processed/links.csv")
code_examples_df = pd.read_csv("data/processed/code_examples.csv")

print("Sections shape:", sections_df.shape)
print("Links shape:", links_df.shape)
print("Code examples shape:", code_examples_df.shape)


## 10. Feature 6 – Documentation Analytics

The project requires at least 8 analytical questions and at least 2 additional questions.

In [ ]:
analysis_results = {}

# 1. How many sections are in the documentation?
analysis_results["total_sections"] = len(sections_df)

# 2. Which section has the highest word count?
highest_word_section = sections_df.loc[sections_df["word_count"].idxmax()]
analysis_results["highest_word_count_section"] = highest_word_section["section_title"]
analysis_results["highest_word_count"] = int(highest_word_section["word_count"])

# 3. Which section contains the most code examples?
code_by_section = code_examples_df.groupby("section_title").size().reset_index(name="code_example_count")
most_code_section = code_by_section.sort_values(by="code_example_count", ascending=False).iloc[0]
analysis_results["most_code_examples_section"] = most_code_section["section_title"]
analysis_results["most_code_examples_count"] = int(most_code_section["code_example_count"])

# 4. Which section contains the most links?
most_links_section = sections_df.loc[sections_df["link_count"].idxmax()]
analysis_results["most_links_section"] = most_links_section["section_title"]
analysis_results["most_links_count"] = int(most_links_section["link_count"])

# 5. What are the top 10 most frequent technical keywords?
technical_keywords = [
    "beautifulsoup", "soup", "tag", "parser", "html", "xml",
    "find", "find_all", "select", "css", "attribute", "text",
    "string", "contents", "children", "parent", "sibling",
    "element", "document", "link"
]

all_text = " ".join(sections_df["section_text"].dropna()).lower()
keyword_counts = []

for keyword in technical_keywords:
    pattern = r"\b" + re.escape(keyword.lower()) + r"\b"
    count = len(re.findall(pattern, all_text))
    keyword_counts.append({"keyword": keyword, "count": count})

keyword_df = pd.DataFrame(keyword_counts)
top_10_keywords = keyword_df.sort_values(by="count", ascending=False).head(10)

# 6. How many internal and external links exist?
link_type_counts = links_df["link_type"].value_counts()
internal_links = int(link_type_counts.get("internal_anchor", 0))
external_links = int(link_type_counts.get("external_link", 0))
analysis_results["internal_links"] = internal_links
analysis_results["external_links"] = external_links

# 7. How many code examples use find_all()?
analysis_results["examples_using_find_all"] = int(code_examples_df["contains_find_all"].sum())

# 8. How many code examples use get_text()?
analysis_results["examples_using_get_text"] = int(code_examples_df["contains_get_text"].sum())

# Additional question 1. What is the average word count per section?
analysis_results["average_word_count_per_section"] = float(np.mean(sections_df["word_count"]))

# Additional question 2. Which code example has the highest line count?
longest_code_example = code_examples_df.loc[code_examples_df["line_count"].idxmax()]
analysis_results["longest_code_example_id"] = int(longest_code_example["example_id"])
analysis_results["longest_code_example_section"] = longest_code_example["section_title"]
analysis_results["longest_code_example_line_count"] = int(longest_code_example["line_count"])

analysis_results


### 10.1 Display Required Analysis Results

In [ ]:
print("1. Total sections:", analysis_results["total_sections"])

print("
2. Section with the highest word count:")
print("Section:", analysis_results["highest_word_count_section"])
print("Word count:", analysis_results["highest_word_count"])

print("
3. Section with the most code examples:")
print("Section:", analysis_results["most_code_examples_section"])
print("Code examples:", analysis_results["most_code_examples_count"])

print("
4. Section with the most links:")
print("Section:", analysis_results["most_links_section"])
print("Links:", analysis_results["most_links_count"])

print("
5. Top 10 most frequent technical keywords:")
display(top_10_keywords)

print("
6. Link type counts:")
display(link_type_counts.reset_index().rename(columns={"index": "link_type", "link_type": "count"}))

print("
Internal links:", analysis_results["internal_links"])
print("External links:", analysis_results["external_links"])

print("
7. Code examples using find_all():", analysis_results["examples_using_find_all"])
print("8. Code examples using get_text():", analysis_results["examples_using_get_text"])

print("
Additional 1. Average word count per section:", round(analysis_results["average_word_count_per_section"], 2))

print("
Additional 2. Longest code example:")
print("Example ID:", analysis_results["longest_code_example_id"])
print("Section:", analysis_results["longest_code_example_section"])
print("Line count:", analysis_results["longest_code_example_line_count"])


## 11. Feature 7 – Data Visualization

Required charts:

1. Bar chart: Top 10 sections by word count  
2. Bar chart: Number of code examples by section  
3. Pie chart: Link type distribution  
4. Histogram: Code example line count distribution  

All charts are saved in `output/charts/`.

In [ ]:
# Chart 1: Top 10 sections by word count

top_sections = sections_df.sort_values(by="word_count", ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.bar(top_sections["section_title"], top_sections["word_count"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Section Title")
plt.ylabel("Word Count")
plt.title("Top 10 Sections by Word Count")
plt.tight_layout()
plt.savefig("output/charts/word_count_by_section.png", dpi=300)
plt.show()


In [ ]:
# Chart 2: Number of code examples by section

code_by_section = code_examples_df.groupby("section_title").size().reset_index(name="code_example_count")
top_code_sections = code_by_section.sort_values(by="code_example_count", ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.bar(top_code_sections["section_title"], top_code_sections["code_example_count"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Section Title")
plt.ylabel("Number of Code Examples")
plt.title("Code Examples by Section")
plt.tight_layout()
plt.savefig("output/charts/code_examples_by_section.png", dpi=300)
plt.show()


In [ ]:
# Chart 3: Link type distribution

link_type_counts = links_df["link_type"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(link_type_counts, labels=link_type_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Link Type Distribution")
plt.tight_layout()
plt.savefig("output/charts/link_type_distribution.png", dpi=300)
plt.show()


In [ ]:
# Chart 4: Code example line count distribution

plt.figure(figsize=(10, 6))
plt.hist(code_examples_df["line_count"], bins=20)
plt.xlabel("Line Count")
plt.ylabel("Frequency")
plt.title("Code Example Line Count Distribution")
plt.tight_layout()
plt.savefig("output/charts/code_linecount_hist.png", dpi=300)
plt.show()


## 12. Extracted Data Summary

In [ ]:
summary_data = {
    "Dataset": ["sections.csv", "links.csv", "code_examples.csv"],
    "Rows": [len(sections_df), len(links_df), len(code_examples_df)],
    "Columns": [sections_df.shape[1], links_df.shape[1], code_examples_df.shape[1]]
}

summary_df = pd.DataFrame(summary_data)
summary_df


## 13. Final Report Content

### 13.1 Dataset Overview

This project analyzes the official BeautifulSoup documentation page. The original dataset is a raw HTML page downloaded directly from the target website. After parsing the page, the system creates three structured datasets: sections, links, and code examples.

### 13.2 Scraping Method

The system uses `requests` to send an HTTP request to the target URL. If the status code is 200, the raw HTML is saved to `data/raw/beautifulsoup_doc.html`. Then, BeautifulSoup parses the HTML content and extracts useful information from headings, links, and code blocks.

### 13.3 Extracted Data Summary

The project generates three processed CSV files:

- `sections.csv`: stores documentation section details.
- `links.csv`: stores hyperlink information and link classifications.
- `code_examples.csv`: stores extracted code examples and related indicators.

### 13.4 Analysis Results

The notebook answers the required analytical questions, including the total number of sections, the section with the highest word count, the section with the most code examples, the section with the most links, the top 10 technical keywords, link type counts, and code example usage of `find_all()` and `get_text()`.

### 13.5 Charts

The notebook creates four required charts:

1. Top 10 sections by word count.
2. Number of code examples by section.
3. Link type distribution.
4. Code example line count distribution.

### 13.6 Key Findings

- The documentation is divided into many structured sections.
- Some sections contain significantly more text than others.
- Code examples are not evenly distributed across all sections.
- Internal anchors are useful for navigating the documentation page.
- BeautifulSoup-related methods such as `find_all()` and `get_text()` appear in code examples.

### 13.7 Limitations

- The project analyzes only one documentation page.
- Link classification is based on rule-based logic.
- Keyword frequency is calculated from a predefined keyword list.
- If the documentation website changes in the future, the results may also change.

### 13.8 Conclusion

This project successfully builds a Python-based documentation analytics system. It collects raw HTML data, parses the documentation, extracts structured datasets, analyzes the extracted data using Pandas and NumPy, creates visualizations using Matplotlib, and prepares final report content in Jupyter Notebook.


## 14. Submission Checklist

Before submitting, make sure these files are available:

- `FinalProject_BeautifulSoup_Analysis.ipynb`
- `data/raw/beautifulsoup_doc.html`
- `data/processed/sections.csv`
- `data/processed/links.csv`
- `data/processed/code_examples.csv`
- `output/charts/word_count_by_section.png`
- `output/charts/code_examples_by_section.png`
- `output/charts/link_type_distribution.png`
- `output/charts/code_linecount_hist.png`
- `README.md`
- `requirements.txt`

To export the notebook to PDF, use Jupyter Notebook menu:

`File` → `Save and Export Notebook As` → `PDF`